Plantas do Brasil

**Fonte:** [GBIF — Global Biodiversity Information Facility](https://www.gbif.org/)  
**Escopo:** Registros de ocorrência de plantas com flor nativas do Brasil, com coordenadas geográficas validadas.

Para auxiliar a visualizacao dos dados obtidos pela analise, houve a criacao de mapa, construido em camadas ativaveis/desativaveis pelo painel de controle


In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster, MiniMap, Fullscreen
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import import_ipynb
from analise import k_ideal
import warnings
import os

warnings.filterwarnings("ignore")

### 1. Paleta de Cores e Configuracao

As cores do popup e da legenda sao extraidas das CSS variables do portfolio -
mesma identidade visual dos graficos da Etapa 2.

O mapa usa o tile `CartoDB dark_matter`, que combina com a paleta verde escura.

In [2]:
PORTFOLIO = {
    "bg":      "#071a0e",
    "bg2":     "#0c2214",
    "card":    "#0f2918",
    "border":  "#2a6642",
    "accent":  "#5ecf8a",
    "accent2": "#3aad6a",
    "accent3": "#a8edca",
    "text":    "#d6f5e3",
    "muted":   "#7bbf9a",
    "link":    "#7de8ac",
}

CORES_CLUSTER = {
    "0": "#ff007f",
    "1": "#ff7675",
    "2": "#ffeaa7",
    "3": "#00cec9",
    "4": "#df80ff",
}

NOMES_CLUSTER = {
    "0": "Cluster A",
    "1": "Cluster B",
    "2": "Cluster C",
    "3": "Cluster D",
    "4": "Cluster E",
}

### 2. Carregamento e Preparacao dos Dados

Alem de carregar o parquet, esta etapa re-executa o K-Means com  K ideal da Etapa 2.

A amostra para marcadores individuais foi feita para evitar problemas com lentidão. O HeatMap em si usa todos os registros 

In [3]:
CAMINHO_DADOS = "dados/plantas_brasil.parquet"
TAMANHO_AMOSTRA = 2000
K_CLUSTERS = k_ideal

df = pd.read_parquet(CAMINHO_DADOS)
df = df.dropna(subset=["latitude", "longitude", "especie"]).copy()

print(f"Registros com coordenadas validas: {len(df):,}")

Registros com coordenadas validas: 8,348


In [4]:
scaler = StandardScaler()
X = scaler.fit_transform(df[["latitude", "longitude"]])

km = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init=10)
df["cluster"] = km.fit_predict(X).astype(str)

print(f"Clusters calculados (K={K_CLUSTERS}):")
print(df["cluster"].value_counts().sort_index().to_string())

Clusters calculados (K=5):
cluster
0    2542
1    1442
2    3507
3     200
4     657


In [5]:
df_amostra = df.sample(n=min(TAMANHO_AMOSTRA, len(df)), random_state=42)

print(f"Total para heatmap  : {len(df):,} pontos")
print(f"Amostra para marcadores: {len(df_amostra):,} pontos")

Total para heatmap  : 8,348 pontos
Amostra para marcadores: 2,000 pontos


### 3. Mapa Base

O Folium gera mapas interativos em HTML via Leaflet.js.
O arquivo final e um `.html` standalone

In [6]:
mapa = folium.Map(
    location=[-14.0, -51.0],      
    zoom_start=4,
    tiles=None,  
    control_scale=True,
)

folium.TileLayer("CartoDB dark_matter", name="Modo Escuro").add_to(mapa)
folium.TileLayer("CartoDB positron", name="Modo Claro").add_to(mapa)

### 4. Camada de Calor 

O HeatMap mostra densidade de ocorrencias: regioes com mais registros aparecem
mais intensas

In [7]:
coordenadas = df[["latitude", "longitude"]].values.tolist()

heatmap_layer = folium.FeatureGroup(name="Densidade de Ocorrencias (Heatmap)")

HeatMap(
    coordenadas,
    radius=12,
    blur=15,
    min_opacity=0.3,
    gradient={
        "0.2": PORTFOLIO["bg2"],      
        "0.4": PORTFOLIO["border"],   
        "0.6": PORTFOLIO["accent2"],
        "0.8": PORTFOLIO["muted"],   
        "1.0": PORTFOLIO["accent3"], 
    }
).add_to(heatmap_layer)

heatmap_layer.add_to(mapa)

### 5. Marcadores Interativos

`MarkerCluster` agrupa automaticamente marcadores proximos em circulos numerados.
Ao dar zoom, os grupos se abrem em pontos individuais, cada um com popup contendo
as informacoes da ocorrencia.

In [8]:
def montar_popup(row: pd.Series) -> str:
    cor = CORES_CLUSTER.get(str(row["cluster"]), PORTFOLIO["accent"])

    estado    = row.get("estado",    "N/A") if pd.notna(row.get("estado"))    else "N/A"
    municipio = row.get("municipio", "N/A") if pd.notna(row.get("municipio")) else "N/A"
    ano       = int(row["ano"])              if pd.notna(row.get("ano"))       else "N/A"
    familia   = row.get("familia",   "N/A") if pd.notna(row.get("familia"))   else "N/A"
    nome_cluster = NOMES_CLUSTER.get(str(row["cluster"]), f"Cluster {row['cluster']}")

    return f"""
    <div style="
        font-family: monospace;
        background: {PORTFOLIO['card']};
        color: {PORTFOLIO['text']};
        padding: 12px 14px;
        border-left: 3px solid {cor};
        min-width: 200px;
        font-size: 11px;
        line-height: 1.8;
    ">
        <div style="color:{cor}; font-size:13px; font-weight:bold; margin-bottom:8px;">
            {row['especie']}
        </div>
        <b style="color:{PORTFOLIO['muted']}">Familia</b>   {familia}<br>
        <b style="color:{PORTFOLIO['muted']}">Estado</b>    {estado}<br>
        <b style="color:{PORTFOLIO['muted']}">Municipio</b> {municipio}<br>
        <b style="color:{PORTFOLIO['muted']}">Ano</b>       {ano}<br>
        <b style="color:{PORTFOLIO['muted']}">Cluster</b>   {nome_cluster}<br>
        <b style="color:{PORTFOLIO['muted']}">Lat/Long</b>  {row['latitude']:.4f}, {row['longitude']:.4f}
    </div>
    """

In [9]:
marcadores_layer = folium.FeatureGroup(name="Registros Individuais (clique para info)")
cluster_group    = MarkerCluster().add_to(marcadores_layer)

for _, row in df_amostra.iterrows():
    cor = CORES_CLUSTER.get(str(row["cluster"]), PORTFOLIO["accent"])

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.7,
        weight=1,
        popup=folium.Popup(montar_popup(row), max_width=260),
        tooltip=row["especie"],
    ).add_to(cluster_group)

marcadores_layer.add_to(mapa)

### 6. Camadas por Cluster

Alem dos marcadores gerais, cada cluster recebe uma `FeatureGroup` propria.
Isso permite ligar e desligar cada grupo individualmente no painel de camadas. As camadas comecam desativadas (`show=False`) para nao sobrecarregar o mapa
na carga inicial.

In [10]:
for cluster_id in sorted(df_amostra["cluster"].unique()):
    cor    = CORES_CLUSTER.get(str(cluster_id), PORTFOLIO["accent"])
    nome   = NOMES_CLUSTER.get(str(cluster_id), f"Cluster {cluster_id}")
    grupo  = df_amostra[df_amostra["cluster"] == cluster_id]

    layer = folium.FeatureGroup(
        name=f"<span style='color:{cor}'>&#9679;</span> {nome} ({len(grupo)} pts)",
        show=False
    )

    for _, row in grupo.iterrows():
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=4,
            color=cor,
            fill=True,
            fill_color=cor,
            fill_opacity=0.8,
            weight=0,
            tooltip=f"{row['especie']} | {row.get('estado', '')}",
        ).add_to(layer)

    layer.add_to(mapa)
    print(f"  {nome}: {len(grupo)} pontos | cor {cor}")

print("Camadas por cluster adicionadas.")

  Cluster A: 624 pontos | cor #ff007f
  Cluster B: 309 pontos | cor #ff7675
  Cluster C: 856 pontos | cor #ffeaa7
  Cluster D: 53 pontos | cor #00cec9
  Cluster E: 158 pontos | cor #df80ff
Camadas por cluster adicionadas.


### 7. Plugins e Painel de Camadas

Tres elementos de navegacao:

- **MiniMap** — minimapa no canto inferior direito para orientacao geral
- **Fullscreen** — botao de tela cheia no canto superior direito
- **LayerControl** — painel que lista todas as FeatureGroups para toggle

In [11]:
MiniMap(
    position="bottomright",
    width=150, height=150,
    collapsed_width=20, collapsed_height=20,
    zoom_level_offset=-5,
).add_to(mapa)

Fullscreen(position="topright").add_to(mapa)

folium.LayerControl(collapsed=False, position="topleft").add_to(mapa)

### 8. Exportacao

O mapa e salvo como um unico arquivo `.html`

In [12]:
CAMINHO_SAIDA = "mapa_plantas_brasil.html"

mapa.save(CAMINHO_SAIDA)

tamanho_mb = os.path.getsize(CAMINHO_SAIDA) / (1024 * 1024)


print(f"  Arquivo: {CAMINHO_SAIDA}")
print(f"  Tamanho: {tamanho_mb:.1f} MB")
print(f"  Heatmap: {len(df):>6,} registros")
print(f"  Marcadores: {len(df_amostra):>6,} pontos interativos")
print(f"  Clusters: {df_amostra['cluster'].nunique()} camadas com toggle")
print(f"  macOS: open {CAMINHO_SAIDA}")
print(f"  Linux: xdg-open {CAMINHO_SAIDA}")
print(f"  Windows: clique duplo no arquivo")

  Arquivo: mapa_plantas_brasil.html
  Tamanho: 5.5 MB
  Heatmap:  8,348 registros
  Marcadores:  2,000 pontos interativos
  Clusters: 5 camadas com toggle
  macOS: open mapa_plantas_brasil.html
  Linux: xdg-open mapa_plantas_brasil.html
  Windows: clique duplo no arquivo
